# PLUMED `pesmd` Validation For Muller-Brown MetaD

This notebook is a reference sanity-check for the NumPy active-bias implementation in `14_muller_brown_active_bias.py`. It does not replace the NumPy Muller-Brown experiment. It uses PLUMED's `pesmd` engine to run Langevin dynamics directly on a two-dimensional potential energy surface, then compares the resulting sampled distribution with an analogous NumPy metadynamics run using the same Muller-Brown potential, walls, hill height, hill width, and hill deposition pace.

Official PLUMED docs describe `pesmd` as biased Langevin dynamics on a 2D potential energy surface, with the potential supplied by a PLUMED input file. The PLUMED Muller-Brown masterclass uses `DISTANCE ... COMPONENTS`, lower/upper harmonic walls, and `METAD` on the two coordinate components.

## 1. Colab Setup

This notebook is intentionally isolated from Poetry. PLUMED is installed through `micromamba` from `conda-forge` because the package is not consistently available through Ubuntu `apt` in Colab.

In [ ]:
from pathlib import Path
import os
import shlex
import shutil
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB and not Path("pyproject.toml").exists():
    !git clone https://github.com/Komputerowe-Projektowanie-Lekow/pmarlo.git
    %cd pmarlo

if IN_COLAB:
    %pip install -q numpy pandas matplotlib scipy

micromamba_bin = Path("/content/bin/micromamba")
plumed_env = Path("/content/plumed-env")
plumed_on_path = shutil.which("plumed")

if IN_COLAB:
    if not micromamba_bin.exists():
        !mkdir -p /content
        !curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xvj -C /content bin/micromamba

    if not (plumed_env / "bin" / "plumed").exists():
        !/content/bin/micromamba create -y -p /content/plumed-env -c conda-forge plumed

    plumed_cmd = [str(micromamba_bin), "run", "-p", str(plumed_env), "plumed"]
    PLUMED_COMMAND = " ".join(shlex.quote(x) for x in plumed_cmd)

elif plumed_on_path is not None:
    plumed_cmd = [plumed_on_path]
    PLUMED_COMMAND = shlex.quote(plumed_on_path)

else:
    raise RuntimeError("PLUMED executable was not found. In Colab, rerun this setup cell.")

os.environ["PLUMED_COMMAND"] = PLUMED_COMMAND
print(f"Using PLUMED command: {PLUMED_COMMAND}")

version_result = subprocess.run(
    plumed_cmd + ["info", "--version"],
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    check=False,
)

print(version_result.stdout)

if version_result.returncode != 0:
    raise RuntimeError("PLUMED was installed but could not run; see output above.")

help_result = subprocess.run(
    plumed_cmd + ["pesmd", "--help"],
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    check=False,
)

print("\n".join(help_result.stdout.splitlines()[:60]))

## 2. Shared Constants And NumPy Reference Code

The constants are the same as the corrected active-bias NumPy notebook. This is classical non-tempered metadynamics, matching the production Muller-Brown runner. `PLUMED_HEIGHT` defaults to `1.0`, matching the full Muller-Brown protocol: 50 hills over 25000 steps deposits about 50 model energy units against a roughly 38-unit barrier.

In [ ]:
from __future__ import annotations

import importlib.util
import json
import math
import os
import shutil
import subprocess
import sys
import textwrap
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd()
OUTPUT_DIR = PROJECT_ROOT / "example_programs" / "programs_outputs" / "plumed_pesmd_validation_colab"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

module_path = PROJECT_ROOT / "example_programs" / "14_muller_brown_active_bias.py"

module_name = "mb_active"
spec = importlib.util.spec_from_file_location(module_name, module_path)

if spec is None or spec.loader is None:
    raise RuntimeError(f"Could not load module from {module_path}")

mb = importlib.util.module_from_spec(spec)
sys.modules[module_name] = mb

try:
    spec.loader.exec_module(mb)
except Exception:
    sys.modules.pop(module_name, None)
    raise

NSTEP = 25_000
PRINT_STRIDE = 10
PLUMED_SEED = 20260518
PLUMED_HEIGHT = 1.0
PLUMED_SIGMA = 0.1
PLUMED_PACE = 500

PLUMED_BIN = os.environ.get("PLUMED_BIN") or shutil.which("plumed")
if PLUMED_BIN is None:
    raise RuntimeError(
        "PLUMED executable was not found. Rerun the setup cell that installs PLUMED through micromamba."
    )

mb.assert_muller_brown_stationary_energies()
reference_probability, xedges, yedges = mb.mb_reference_probability()
print(f"Output directory: {OUTPUT_DIR}")

## 3. Write PLUMED `pesmd` Inputs

`pesmd.in` controls the reduced Langevin simulation. `plumed.dat` defines the Muller-Brown potential through `MATHEVAL`, adds harmonic walls, applies `METAD` to `(x,y)`, and prints `COLVAR`.

In [ ]:
MB_FUNC = (
    "-200*exp(-1*(x-1)^2-10*(y-0)^2)"
    "-100*exp(-1*(x-0)^2-10*(y-0.5)^2)"
    "-170*exp(-6.5*(x+0.5)^2+11*(x+0.5)*(y-1.5)-6.5*(y-1.5)^2)"
    "+15*exp(0.7*(x+1)^2+0.6*(x+1)*(y-1)+0.7*(y-1)^2)"
)

plumed_dat = f"""
cv: DISTANCE ATOMS=1,2 COMPONENTS
ff: MATHEVAL ARG=cv.x,cv.y FUNC={MB_FUNC} PERIODIC=NO
bb: BIASVALUE ARG=ff
lwall: LOWER_WALLS ARG=cv.x,cv.y AT={mb.WALL_X_MIN},{mb.WALL_Y_MIN} KAPPA={mb.WALL_K},{mb.WALL_K}
uwall: UPPER_WALLS ARG=cv.x,cv.y AT={mb.WALL_X_MAX},{mb.WALL_Y_MAX} KAPPA={mb.WALL_K},{mb.WALL_K}
metad: METAD ARG=cv.x,cv.y SIGMA={PLUMED_SIGMA},{PLUMED_SIGMA} HEIGHT={PLUMED_HEIGHT} PACE={PLUMED_PACE} FILE=HILLS GRID_MIN={mb.WALL_X_MIN},{mb.WALL_Y_MIN} GRID_MAX={mb.WALL_X_MAX},{mb.WALL_Y_MAX}
PRINT ARG=cv.x,cv.y,ff,metad.bias FILE=COLVAR STRIDE={PRINT_STRIDE}
""".strip()

pesmd_in = f"""
temperature {mb.LANGEVIN_KT}
tstep {mb.LANGEVIN_DT}
friction {mb.LANGEVIN_GAMMA}
dimension 2
nstep {NSTEP}
ipos {mb.MB_INIT_BASIN[0]} {mb.MB_INIT_BASIN[1]}
idum {PLUMED_SEED}
plumed plumed.dat
""".strip()

(OUTPUT_DIR / "plumed.dat").write_text(plumed_dat + "\n", encoding="utf-8")
(OUTPUT_DIR / "pesmd.in").write_text(pesmd_in + "\n", encoding="utf-8")
print((OUTPUT_DIR / "plumed.dat").read_text(encoding="utf-8"))
print("--- pesmd.in ---")
print((OUTPUT_DIR / "pesmd.in").read_text(encoding="utf-8"))

## 4. Run PLUMED `pesmd`

This cell fails loudly if PLUMED cannot parse the input or does not produce `COLVAR` and `HILLS`.

In [ ]:
from pathlib import Path
import os
import shutil
import subprocess
import time

# Preferuj komendę ustawioną w komórce instalacyjnej:
# /content/bin/micromamba run -p /content/plumed-env plumed
PLUMED_COMMAND = os.environ.get("PLUMED_COMMAND")

if PLUMED_COMMAND is None:
    plumed_on_path = shutil.which("plumed")
    if plumed_on_path is None:
        raise RuntimeError(
            "PLUMED executable was not found. "
            "Rerun the setup cell that installs PLUMED through micromamba."
        )
    PLUMED_COMMAND = plumed_on_path

print(f"Using PLUMED command: {PLUMED_COMMAND}")

pesmd_input_path = OUTPUT_DIR / "pesmd.in"

if not pesmd_input_path.exists():
    raise RuntimeError(f"Missing input file: {pesmd_input_path}")

# Workaround na crash PLUMED pesmd:
# w niektórych wersjach pesmd wymaga jawnie wpisanej flagi periodic.
pesmd_text = pesmd_input_path.read_text(encoding="utf-8")

has_periodic_key = any(
    line.strip()
    and not line.strip().startswith("#")
    and line.strip().split()[0].lower() == "periodic"
    for line in pesmd_text.splitlines()
)

if not has_periodic_key:
    pesmd_text = pesmd_text.rstrip() + "\nperiodic false\n"
    pesmd_input_path.write_text(pesmd_text, encoding="utf-8")
    print("Added missing line to pesmd.in: periodic false")

print("\nCurrent pesmd.in:")
print(pesmd_input_path.read_text(encoding="utf-8"))


def run_command(command: str, cwd: Path) -> subprocess.CompletedProcess[str]:
    print(f"\nRunning command:\n{command}\n")

    completed = subprocess.run(
        command,
        cwd=cwd,
        shell=True,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        check=False,
    )

    log_path = cwd / "plumed_pesmd.log"
    log_path.write_text(completed.stdout, encoding="utf-8")

    print(completed.stdout[-4000:])

    if completed.returncode != 0:
        raise RuntimeError(
            f"Command failed with code {completed.returncode}; see {log_path}"
        )

    return completed


# Usuń stare pliki wyjściowe, żeby nie pomylić nowego wyniku ze starym.
for filename in [
    "COLVAR",
    "HILLS",
    "bck.0.COLVAR",
    "bck.0.HILLS",
    "stats.out",
    "plumed_pesmd.log",
]:
    path = OUTPUT_DIR / filename
    if path.exists():
        path.unlink()

started = time.perf_counter()

# Ważne: nie dawaj cudzysłowu wokół całego PLUMED_COMMAND,
# bo w Colabie to może być komenda wieloczłonowa przez micromamba.
run_command(f"{PLUMED_COMMAND} pesmd < pesmd.in", OUTPUT_DIR)

elapsed = time.perf_counter() - started
print(f"PLUMED pesmd elapsed seconds: {elapsed:.2f}")

assert (OUTPUT_DIR / "COLVAR").exists(), "PLUMED did not write COLVAR"
assert (OUTPUT_DIR / "HILLS").exists(), "PLUMED did not write HILLS"

print("PLUMED pesmd finished successfully.")

## 5. Run Analogous NumPy MetaD On `(x,y)`

This is not the adaptive retraining experiment. It is only the same fixed-CV MetaD protocol as the PLUMED `pesmd` run, used to check that the local bias implementation is in the same regime.

In [ ]:
class IdentityCVModel:
    def transform(self, xy):
        return np.asarray(xy, dtype=np.float64)

    def jacobian(self, xy=None):
        return np.eye(2, dtype=np.float64)


def run_numpy_metad() -> tuple[np.ndarray, np.ndarray]:
    rng = np.random.default_rng(PLUMED_SEED)
    velocity = rng.normal(scale=math.sqrt(mb.LANGEVIN_KT / mb.LANGEVIN_MASS), size=2)
    sim = mb.LangevinMB(mb.MB_INIT_BASIN.copy(), velocity, rng)
    model = IdentityCVModel()
    ledger = mb.ActiveBiasLedger(PLUMED_SIGMA, PLUMED_HEIGHT)
    trajectory = np.empty((NSTEP, 2), dtype=np.float64)
    bias_values = np.empty(NSTEP, dtype=np.float64)

    def force_fn(xy):
        return ledger.force_on_xy(xy, model)

    for step in range(NSTEP):
        xy = sim.step(force_fn)
        trajectory[step] = xy
        bias_values[step] = ledger.potential_in_cv(model.transform(xy))
        if step % PLUMED_PACE == 0:
            ledger.add_hill(xy, model)
    return trajectory[::PRINT_STRIDE], bias_values[::PRINT_STRIDE]


started = time.perf_counter()
numpy_xy, numpy_bias = run_numpy_metad()
print(f"NumPy MetaD elapsed seconds: {time.perf_counter() - started:.2f}")
pd.DataFrame({"x": numpy_xy[:, 0], "y": numpy_xy[:, 1], "bias": numpy_bias}).to_csv(
    OUTPUT_DIR / "numpy_metad_colvar.csv",
    index=False,
)

## 6. Load PLUMED Outputs And Compare Distributions

In [ ]:
def read_colvar(path: Path) -> pd.DataFrame:
    with path.open(encoding="utf-8") as handle:
        header = None
        for line in handle:
            if line.startswith("#! FIELDS"):
                header = line.split()[2:]
                break
    if header is None:
        raise ValueError(f"Could not find FIELDS header in {path}")
    data = pd.read_csv(path, comment="#", delim_whitespace=True, names=header)
    return data


def histogram_probability(points: np.ndarray, pseudocount: float = 1.0e-12) -> np.ndarray:
    hist, _, _ = np.histogram2d(points[:, 0], points[:, 1], bins=(xedges, yedges))
    hist = hist.astype(np.float64) + pseudocount
    hist /= float(hist.sum())
    return hist


def kl_divergence(p_ref: np.ndarray, p_est: np.ndarray) -> float:
    return float(np.sum(p_ref * (np.log(p_ref) - np.log(p_est))))


plumed_colvar = read_colvar(OUTPUT_DIR / "COLVAR")
plumed_xy = plumed_colvar[["cv.x", "cv.y"]].to_numpy(dtype=np.float64)
numpy_probability = histogram_probability(numpy_xy)
plumed_probability = histogram_probability(plumed_xy)

metrics = {
    "plumed_frames": int(plumed_xy.shape[0]),
    "numpy_frames": int(numpy_xy.shape[0]),
    "plumed_kl_ref_raw": kl_divergence(reference_probability, plumed_probability),
    "numpy_kl_ref_raw": kl_divergence(reference_probability, numpy_probability),
    "plumed_numpy_l1_hist": float(np.sum(np.abs(plumed_probability - numpy_probability))),
    "plumed_hills": int(sum(1 for line in (OUTPUT_DIR / "HILLS").read_text(encoding="utf-8").splitlines() if line and not line.startswith("#!"))),
}
(OUTPUT_DIR / "plumed_pesmd_validation_metrics.json").write_text(json.dumps(metrics, indent=2), encoding="utf-8")
metrics

## 7. Plot The Sanity Check

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].imshow(-np.log(reference_probability.T), origin="lower", extent=(mb.WALL_X_MIN, mb.WALL_X_MAX, mb.WALL_Y_MIN, mb.WALL_Y_MAX), aspect="auto")
axes[0].set_title("Analytic reference")

axes[1].plot(plumed_xy[:, 0], plumed_xy[:, 1], lw=0.5, alpha=0.8)
axes[1].set_title("PLUMED pesmd trajectory")

axes[2].plot(numpy_xy[:, 0], numpy_xy[:, 1], lw=0.5, alpha=0.8)
axes[2].set_title("NumPy MetaD trajectory")

for ax in axes:
    ax.set_xlim(mb.WALL_X_MIN, mb.WALL_X_MAX)
    ax.set_ylim(mb.WALL_Y_MIN, mb.WALL_Y_MAX)
    ax.set_xlabel("x")
    ax.set_ylabel("y")

fig.tight_layout()
plot_path = OUTPUT_DIR / "plumed_pesmd_validation.png"
fig.savefig(plot_path, dpi=180)
print(f"Saved {plot_path}")

## Interpretation

This notebook is not expected to produce identical trajectories. PLUMED `pesmd` and the NumPy BAOAB code use different random streams and may use different Langevin discretization details. The validation target is weaker and more appropriate: both runs should stay in the same wall-bounded Muller-Brown region, deposit the expected number of hills, produce finite KL metrics, and reach comparable sampled regions/FES within sampling noise. With `HEIGHT=1.0`, this validation should be repeated after the corrected NumPy protocol because it is no longer intentionally sub-threshold. If PLUMED diverges or the histogram comparison is wildly different, inspect `plumed.dat`, `pesmd.in`, `COLVAR`, `HILLS`, and `plumed_pesmd.log` in the output directory.